# Experiment Notebook: Imitation Warm-Start + Double-Q

Goal: initialize from SRTF demonstrations, then fine-tune with Double-Q to reduce value overestimation and improve hard-slice ranking.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str((Path.cwd().parent / "src").resolve()))
from ml_engine import research_upgrade_lab as lab

lab.set_seed(42)
df = lab.load_clean_dataset(Path.cwd())
weights = {"carbon": 0.35, "jct": 0.20, "tail": 0.25, "preempt": 0.10, "starvation": 0.10}

# Reference model
q_ref = lab.train_q_domain_randomized(df=df, episodes=180, reward_weights=weights, hard_bias=0.65)

# Imitation + Double-Q
demo_counts = lab.collect_srtf_demo_counts(df=df, episodes=100)
q_init = demo_counts / np.maximum(1.0, demo_counts.sum(axis=-1, keepdims=True))
q_double = lab.train_double_q(df=df, episodes=240, reward_weights=weights, q_init=q_init)

policy_tables = {
    "srtf": None,
    "q_ref": q_ref,
    "q_doubleq_imit": q_double,
    "carbon": None,
    "fcfs": None,
}

detail = lab.evaluate_policies(
    df=df,
    policy_tables=policy_tables,
    seeds=list(range(20)),
    noises=[15.0, 20.0],
    congestions=["moderate", "high"],
    capacities=[6, 8],
    reward_weights=weights,
)

scored = lab.compute_overall_score_robust(detail, weights)
hard = lab.summarize_hard_slice(scored)
display(hard.round(4))